In [18]:
import os
import json
import glob
import re
import datetime
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
import tkinter as tk
from tkinter import filedialog, messagebox

pio.renderers.default = "notebook"

In [19]:
class KingKevAnalyzer:
    def __init__(self, folders, config, show_nyquist=False):
        self.folders = folders # Now accepts a list
        self.config = config
        self.show_nyquist = show_nyquist
        self.log_paths = [] # Keep track of logs
        
        self.test_name = "Unknown_Test"
        self.testbench = "70804 / KingKev"
        
        self.channels = {i: {
            'id': f'Unknown_CH{i}', 
            'area': 1.0, 
            'true_ocv': None, 
            'df': pd.DataFrame(),
            'ocv_by_hum': {30: '-', 50: '-', 70: '-', 90: '-'}, 
            'summary_data': {}                                  
        } for i in range(1, 5)}
        
        self.timeline = []
        self.figures = {}
        self.colors = self.config.get("custom_colors", px.colors.qualitative.Plotly) * 3

    def _find_true_ocv(self, ch_num, log_path):
        if not log_path: 
            return
        try:
            with open(log_path, 'r', encoding='latin1') as f:
                lines = f.readlines()
            
            found_ocv = None
            start_str = f"Sub Eis_Ocv_Channel_{ch_num}, START"
            start_index = next(i for i, line in enumerate(lines) if start_str in line)
            
            voltage_pattern = re.compile(rf'voltage CH0{ch_num}:\s*([\d,.]+)')
            for i in range(start_index, len(lines)):
                match = voltage_pattern.search(lines[i])
                if match:
                    found_ocv = float(match.group(1).replace(',', '.'))
                    break
        except StopIteration:
            found_ocv = None
            
        self.channels[ch_num]['true_ocv'] = found_ocv
        if found_ocv:
            print(f"  -> Found definitive OCV for CH {ch_num}: {found_ocv:.5f} V")

        
    def parse_master_log(self):
        print("\n--- 1. Parsing KingKev Master Log(s) ---")
        metadata_extracted = False
        
        for folder in self.folders:
            log_files = glob.glob(os.path.join(folder, "*LogFile*.txt")) + glob.glob(os.path.join(folder, "*.log"))
            if not log_files: continue
                
            log_path = log_files[0]
            self.log_paths.append(log_path)
            
            with open(log_path, 'r', encoding='latin1') as f: 
                content = f.read()
            
            if not metadata_extracted:
                test_name_match = re.search(r'Test-Name:\s*(.*)', content, re.IGNORECASE)
                if test_name_match: self.test_name = test_name_match.group(1).strip()
                    
                for i in range(1, 5):
                    pattern = rf'Cell-Name-Channel {i}:\s*([A-Za-z0-9_-]+),\s*Area:\s*([\d,.]+)'
                    match = re.search(pattern, content, re.IGNORECASE)
                    if match:
                        self.channels[i]['id'] = match.group(1).strip()
                        self.channels[i]['area'] = float(match.group(2).replace(',', '.'))
                metadata_extracted = True
            
            for i in range(1, 5):
                if self.channels[i]['true_ocv'] is None:
                    self._find_true_ocv(i, log_path)

            env_pattern = re.compile(r'(\d{2}\.\d{2}\.\d{4}\s\d{2}:\d{2}:\d{2})pH2O\s*=\s*(\d+)%,\s*T=([\d.,]+)')
            for match in env_pattern.finditer(content):
                dt_str, hum_str, temp_str = match.groups()
                dt = datetime.datetime.strptime(dt_str, "%d.%m.%Y %H:%M:%S")
                hum = int(hum_str)
                raw_temp = float(temp_str.replace(',', '.'))
                clean_temp = int(round(raw_temp / 25.0) * 25.0) 
                self.timeline.append({'timestamp': dt, 'type': 'env', 'temperature': clean_temp, 'humidity': hum})
                
            pol_start_pattern = re.compile(r'\$?(\d{2}\.\d{2}\.\d{4}\s\d{2}:\d{2}:\d{2}), Sub Pol_Channel_(\d+), START')
            for match in pol_start_pattern.finditer(content):
                dt = datetime.datetime.strptime(match.group(1), "%d.%m.%Y %H:%M:%S")
                ch = int(match.group(2))
                self.timeline.append({'timestamp': dt, 'type': 'pol_start', 'ch': ch})

            eis_start_pattern = re.compile(r'\$?(\d{2}\.\d{2}\.\d{4}\s\d{2}:\d{2}:\d{2}), Sub Eis_Ocv_Channel_(\d+), START')
            for match in eis_start_pattern.finditer(content):
                dt = datetime.datetime.strptime(match.group(1), "%d.%m.%Y %H:%M:%S")
                ch = int(match.group(2))
                self.timeline.append({'timestamp': dt, 'type': 'eis_start', 'ch': ch})
                    
            current_temp, current_hum, current_eis_ch = None, None, None
            for line in content.split('\n'):
                env_match = re.search(r'pH2O\s*=\s*(\d+)%,\s*T=([\d.,]+)', line)
                if env_match:
                    current_hum = int(env_match.group(1))
                    raw_temp = float(env_match.group(2).replace(',', '.'))
                    current_temp = int(round(raw_temp / 25.0) * 25.0)
                
                eis_match = re.search(r'Sub Eis_Ocv_Channel_(\d+), START', line, re.IGNORECASE)
                if eis_match:
                    current_eis_ch = int(eis_match.group(1))
                    
                if current_eis_ch is not None:
                    volt_match = re.search(rf'voltage CH0?{current_eis_ch}:\s*([\d.,]+)V', line, re.IGNORECASE)
                    if volt_match and current_temp == 650:
                        volt_val = float(volt_match.group(1).replace(',', '.'))
                        if current_hum in [30, 50, 70, 90]:
                            if self.channels[current_eis_ch]['ocv_by_hum'][current_hum] == '-':
                                self.channels[current_eis_ch]['ocv_by_hum'][current_hum] = f"{volt_val:.5f}"
                        current_eis_ch = None 

        self.timeline.sort(key=lambda x: x['timestamp'])



    def _extract_resistances_from_eis(self, ch_num):
        """Mathematically interpolates exact Rs and Rtot zero-crossings from the full EIS spectrum"""
        all_eis_files = []
        for folder in self.folders:
            eis_dir = os.path.join(folder, 'Data', 'EIS_orig')
            if not os.path.exists(eis_dir): eis_dir = os.path.join(folder, 'EIS_orig')
            if os.path.exists(eis_dir): all_eis_files.extend(glob.glob(os.path.join(eis_dir, "*.txt")))
                
        if not all_eis_files: return None, None, None, None, None
        
        target_file, target_time = None, None
        
        # 1. Search specifically for the 650°C, 0% Humidity Baseline file
        for file_path in all_eis_files:
            filename = os.path.basename(file_path)
            match = re.search(r'(\d{12})', filename)
            if not match: continue
            
            ts_str = match.group(1)
            try:
                year, month, day = int("20" + ts_str[0:2]), int(ts_str[2:4]), int(ts_str[4:6])
                hour, minute, second = int(ts_str[6:8]), int(ts_str[8:10]), int(ts_str[10:12])
                file_time = datetime.datetime(year, month, day, hour, minute, second)
            except Exception: continue
            
            current_temp, current_hum, current_ch = 650, 0, None
            for event in self.timeline:
                if event['timestamp'] <= file_time:
                    if event['type'] == 'env':
                        current_temp = event['temperature']
                        current_hum = event['humidity']
                    elif event['type'] == 'eis_start':
                        current_ch = event['ch']
                else: break
                    
            ch_match = re.search(r'CH(\d+)', filename, re.IGNORECASE)
            if ch_match: current_ch = int(ch_match.group(1))
                
            if current_ch == ch_num and current_temp == 650 and current_hum == 0:
                if target_time is None or file_time < target_time:
                    target_time, target_file = file_time, file_path
                    
        # 2. Fallback: if no explicit 0% humidity is found, grab the very first curve taken at 650°C
        if not target_file:
            for file_path in all_eis_files:
                filename = os.path.basename(file_path)
                match = re.search(r'(\d{12})', filename)
                if not match: continue
                ts_str = match.group(1)
                try:
                    year, month, day = int("20" + ts_str[0:2]), int(ts_str[2:4]), int(ts_str[4:6])
                    hour, minute, second = int(ts_str[6:8]), int(ts_str[8:10]), int(ts_str[10:12])
                    file_time = datetime.datetime(year, month, day, hour, minute, second)
                except Exception: continue
                
                current_temp, current_ch = 650, None
                for event in self.timeline:
                    if event['timestamp'] <= file_time:
                        if event['type'] == 'env': current_temp = event['temperature']
                        elif event['type'] == 'eis_start': current_ch = event['ch']
                    else: break
                        
                ch_match = re.search(r'CH(\d+)', filename, re.IGNORECASE)
                if ch_match: current_ch = int(ch_match.group(1))
                
                if current_ch == ch_num and current_temp == 650:
                    if target_time is None or file_time < target_time:
                        target_time, target_file = file_time, file_path

        if not target_file: return None, None, None, None, None
        
        try:
            with open(target_file, 'r', encoding='latin1') as file: 
                lines = file.readlines()
            data_start = next((i + 4 for i, l in enumerate(lines) if 'Impedance samples' in l), -1)
            if data_start == -1: return None, None, None, None, None
            
            raw_f, raw_r, raw_i = [], [], []
            for j in range(data_start, len(lines)):
                if lines[j].strip():
                    parts = lines[j].split()
                    if len(parts) >= 4:
                        try:
                            f = float(parts[1])
                            r = float(parts[2])
                            i_val = float(parts[3])
                            if r > 0:
                                raw_f.append(f)
                                raw_r.append(r)         # RAW Ohms Re(Z)
                                raw_i.append(-i_val)    # -Im(Z) (plotted Y-axis)
                        except (ValueError, IndexError): continue
            
            if not raw_r: return None, None, None, None, None
            
            # --- MATHEMATICAL ZERO CROSSING INTERPOLATION ---
            crossings = []
            for idx in range(len(raw_i) - 1):
                y1, y2 = raw_i[idx], raw_i[idx+1]
                x1, x2 = raw_r[idx], raw_r[idx+1]
                
                # Check if the line segment between these two dots crosses y=0 
                if (y1 <= 0 and y2 > 0) or (y1 > 0 and y2 <= 0):
                    if y2 != y1:
                        # Exact linear intersection formula
                        x_cross = x1 + (0 - y1) * (x2 - x1) / (y2 - y1)
                        crossings.append(x_cross)
            
            if len(crossings) >= 2:
                exact_rs = min(crossings)
                exact_rtot = max(crossings)
            elif len(crossings) == 1:
                exact_rs = crossings[0]
                pos_indices = [x for x, y in zip(raw_r, raw_i) if y >= 0]
                exact_rtot = pos_indices[-1] if pos_indices else exact_rs
            else:
                pos_indices = [x for x, y in zip(raw_r, raw_i) if y >= 0]
                exact_rs = pos_indices[0] if pos_indices else None
                exact_rtot = pos_indices[-1] if pos_indices else None
                
            return exact_rs, exact_rtot, raw_f, raw_r, raw_i
        except Exception:
            return None, None, None, None, None
    
    def parse_pol_files(self):
        print("\n--- 2. Parsing & Merging Pol_orig Files (SOFC + SOEC) ---")
        all_pol_files = []
        for folder in self.folders:
            pol_dir = os.path.join(folder, 'Data', 'Pol_orig')
            if not os.path.exists(pol_dir): pol_dir = os.path.join(folder, 'Pol_orig')
            if os.path.exists(pol_dir): all_pol_files.extend(glob.glob(os.path.join(pol_dir, "*.txt")))

        if not all_pol_files: return
        
        raw_data = {1: [], 2: [], 3: [], 4: []}
        
        for file_path in all_pol_files:
            filename = os.path.basename(file_path)
            
            # Match any Channel File (SOFC or SOEC)
            match = re.search(r'CH(\d+).*?_(\d{12})', filename, re.IGNORECASE)
            if not match: continue
            
            ch_num = int(match.group(1))
            if ch_num not in raw_data: continue
            
            ts_str = match.group(2)
            try:
                year, month, day = int("20" + ts_str[0:2]), int(ts_str[2:4]), int(ts_str[4:6])
                hour, minute, second = int(ts_str[6:8]), int(ts_str[8:10]), int(ts_str[10:12])
                file_time = datetime.datetime(year, month, day, hour, minute, second)
            except Exception: continue
            
            # Match to Timeline (Find prevailing Temp/Hum AND the current pol_id for this channel)
            current_temp, current_hum = 650, 50 
            pol_counter = 0
            
            for event in self.timeline:
                if event['timestamp'] <= file_time:
                    if event['type'] == 'env':
                        current_temp = event['temperature']
                        current_hum = event['humidity']
                    elif event['type'] == 'pol_start' and event['ch'] == ch_num:
                        pol_counter += 1
                else: 
                    break
            # Read Data dynamically avoiding header bloat
            try:
                with open(file_path, 'r', encoding='latin1') as f: lines = f.readlines()
                data_start = next((i for i, l in enumerate(lines) if "Potential/V" in l or "Current/A" in l or "UinV" in l or "Voltage" in l), -1)
                
                if data_start == -1: continue
                df = pd.read_csv(file_path, sep=r'\s+', skiprows=data_start, encoding='latin1')
                v_col = next((c for c in df.columns if 'POTENTIAL' in c.upper() or 'UINV' in c.upper() or 'VOLTAGE' in c.upper() or '/V' in c.upper()), None)
                i_col = next((c for c in df.columns if 'CURRENT' in c.upper() or 'IINA' in c.upper() or '/A' in c.upper()), None)
                
                if not v_col or not i_col: continue
                
                # MATHEMATICAL FIX: Force voltage to be positive. Keep Zahner Current signs intact!
                voltage = np.abs(df[v_col].values)
                j_density = df[i_col].values / self.channels[ch_num]['area']
                
                if len(voltage) > 0:
                    raw_data[ch_num].append({
                        'timestamp': file_time,
                        'pol_id': pol_counter,
                        'temperature': current_temp,
                        'humidity': current_hum,
                        'Voltage': voltage,
                        'CurrentDensity': j_density,
                        'filename': filename
                    })
            except Exception: pass
                
        # Merge Files Sharing the Same pol_id (Fusing SOFC and SOEC together)
        total_merged_curves = 0
        for ch_num in range(1, 5):
            df_unmerged = pd.DataFrame(raw_data[ch_num])
            if df_unmerged.empty: continue
            
            merged_pol_data = []
            for pol_id, group in df_unmerged.groupby('pol_id'):
                if pd.isna(pol_id) or pol_id == 0:
                    for _, row in group.iterrows(): merged_pol_data.append(row.to_dict())
                    continue
                    
                combined_v = np.concatenate(group['Voltage'].tolist())
                combined_j = np.concatenate(group['CurrentDensity'].tolist())
                
                # CRITICAL: Sort by Current Density. This ensures the line draws smoothly from SOEC to SOFC!
                sort_idx = np.argsort(combined_j)
                first = group.sort_values('timestamp').iloc[0]
                
                merged_row = first.to_dict()
                merged_row['filename'] = " + ".join(group['filename'].tolist())
                merged_row['Voltage'] = combined_v[sort_idx]
                merged_row['CurrentDensity'] = combined_j[sort_idx]
                merged_pol_data.append(merged_row)
                
            self.channels[ch_num]['df'] = pd.DataFrame(merged_pol_data).sort_values('timestamp')
            print(f"  -> Channel {ch_num} ({self.channels[ch_num]['id']}): Merged into {len(merged_pol_data)} full Continuous curves.")
            total_merged_curves += len(merged_pol_data)
            
        if total_merged_curves == 0:
            print("\n  !!! CRITICAL FAILURE: Zero merged curves loaded. !!!")

    def _preview_table(self, df_subset, plot_title):
        if df_subset.empty: return
        print(f"\n--- Data Preview for: {plot_title} ---")
        display_cols = ['filename', 'timestamp', 'temperature', 'humidity', 'pol_id']
        preview_df = df_subset[display_cols].copy()
        print(preview_df.to_string(index=False))

    def _create_nyquist_figure(self, cell_id, ch_num, f_arr, re_arr, im_arr, rs_norm, rtot_norm, rpol_norm):
        """Generates an interactive Nyquist plot showing ALL points (including inductive tails) and exact crossings"""
        title = f"EIS Nyquist for {cell_id} (CH{ch_num}) at 650°C (100% H₂)"
        fig = go.Figure()
        
        name_str = f"100% H₂ / 0% H₂O<br>(Rs: {rs_norm:.4f}, Rtot: {rtot_norm:.4f}, Rpol: {rpol_norm:.4f})"
        
        # Plot ALL recorded data points, including the inductive tail
        fig.add_trace(go.Scatter(
            x=re_arr, y=im_arr, mode='markers+lines', name=name_str,
            marker=dict(size=8, color='blue', symbol='circle'), line=dict(width=2.5, color='blue'),
            text=f_arr, hovertemplate="<b>Baseline</b><br>Re(Z): %{x:.4f} Ω·cm²<br>-Im(Z): %{y:.4f} Ω·cm²<br>Freq: %{text:.1f} Hz<extra></extra>"
        ))
        
        # Plot physical RED 'X' markers directly on the EXACT interpolated zero-crossings
        fig.add_trace(go.Scatter(
            x=[rs_norm, rtot_norm], y=[0, 0], mode='markers', name="Exact Zero-Crossings (y=0)",
            marker=dict(size=14, color='red', symbol='x', line=dict(width=2, color='black')),
            hoverinfo='skip'
        ))
        
        fig.update_layout(
            title=dict(text=title, font=dict(family="Arial", size=22, color="black")),
            template="plotly_white", height=800, width=1100, margin=dict(l=80, r=80, t=80, b=80),
            legend=dict(x=0.98, y=0.98, xanchor="right", yanchor="top", bordercolor="#888", borderwidth=1, bgcolor="rgba(255,255,255,0.9)"),
            xaxis=dict(title="Re(Z) / Ω·cm²", scaleanchor="y", scaleratio=1), # Forces 1:1 Aspect Ratio
            yaxis=dict(title="-Im(Z) / Ω·cm²")
        )
        
        # Draw the physical x-axis line so you can clearly see the tail crossing it
        fig.add_hline(y=0.0, line_width=1, line_color="black")
        self.figures[title] = fig

    def generate_analyst_plots(self, preview_data=False):
        print("\n--- 3. Generating KingKev Plots & Summary Tables ---")
        
        for ch_num, ch_data in self.channels.items():
            df = ch_data['df']
            cell_id = ch_data['id']
            if df.empty: continue
            
            print(f"\n--- Preparing Table for Channel {ch_num} ({cell_id}) ---")
            
            # AUTOMATICALLY EXTRACT Rs and Rtot using Interpolation
            extracted_rs, extracted_rtot, e_f, e_r, e_i = self._extract_resistances_from_eis(ch_num)
            
            if extracted_rs is not None and extracted_rtot is not None:
                area_norm_rs = extracted_rs * ch_data['area']
                area_norm_rtot = extracted_rtot * ch_data['area']
                area_norm_rpol = area_norm_rtot - area_norm_rs
                
                print(f"  -> Extracted Exact Zero-Crossings:")
                print(f"     Rs:   {extracted_rs:.6f} Ω ({area_norm_rs:.4f} Ω·cm²)")
                print(f"     Rtot: {extracted_rtot:.6f} Ω ({area_norm_rtot:.4f} Ω·cm²)")
                print(f"     Rpol: {(extracted_rtot - extracted_rs):.6f} Ω ({area_norm_rpol:.4f} Ω·cm²)")
                
                # DRAW NYQUIST PLOT IF TAILS MODE IS ENABLED
                if self.show_nyquist and e_r:
                    self._create_nyquist_figure(cell_id, ch_num, e_f, np.array(e_r)*ch_data['area'], np.array(e_i)*ch_data['area'], area_norm_rs, area_norm_rtot, area_norm_rpol)
            else:
                extracted_rs, extracted_rtot = "-", "-"
            
            # 1. Humidity Variation at 650C
            hum_30 = df[(df['temperature'] == 650) & (df['humidity'] == 30)].head(1)
            hum_50_init = df[(df['temperature'] == 650) & (df['humidity'] == 50)].head(1)
            hum_70 = df[(df['temperature'] == 650) & (df['humidity'] == 70)].head(1)
            hum_90 = df[(df['temperature'] == 650) & (df['humidity'] == 90)].head(1)
            
            # 2. Stability Check
            if not hum_90.empty:
                time_90 = hum_90.iloc[0]['timestamp']
                hum_50_post = df[(df['temperature'] == 650) & (df['humidity'] == 50) & (df['timestamp'] > time_90)].head(1)
            else:
                hum_50_post = pd.DataFrame()
                
            # 3. Temperature Variation
            temp_var_df = df[df['humidity'] == 50].copy()
            if not temp_var_df.empty:
                temp_var_df = temp_var_df[temp_var_df['temperature'].isin([750, 725, 700, 675, 650, 625, 600])]
                temp_var_df = temp_var_df.drop_duplicates(subset=['temperature'], keep='last').sort_values('temperature', ascending=False)
                
            # 4. Aging Check
            hum_50_final = df[(df['temperature'] == 650) & (df['humidity'] == 50)].tail(1)
            
            plot_definitions = [
                (f"IV Humidity Variation for {cell_id} at 650°C (CH{ch_num})", pd.concat([hum_30, hum_50_init, hum_70, hum_90]), "humidity"),
                (f"Stability Check of {cell_id} at 650°C after High Humidity Exposure (CH{ch_num})", pd.concat([hum_50_init, hum_50_post]), "stability"),
                (f"IV Temperature Variation for {cell_id} at 50% H₂O (CH{ch_num})", temp_var_df, "temperature"),
                (f"Aging of {cell_id} at 50% H₂O and 650°C (CH{ch_num})", pd.concat([hum_50_init, hum_50_final]), "aging")
            ]
            
            for title, plot_df, plot_type in plot_definitions:
                if plot_df.empty or len(plot_df) < 2: continue
                
                if preview_data: self._preview_table(plot_df, title)
                fig = self._create_base_figure(title)
                
                for i, (_, row) in enumerate(plot_df.iterrows()):
                    if plot_type == "humidity":
                        name = f"{row['humidity']}% H₂O"
                    elif plot_type == "stability":
                        name = "Baseline 50% H₂O" if i == 0 else "Post-High Humidity 50% H₂O"
                    elif plot_type == "temperature":
                        name = f"{row['temperature']}°C"
                    elif plot_type == "aging":
                        name = "Initial Reference (BoT)" if i == 0 else "Final Check (EoT)"
                        
                    self._add_trace(fig, row, name, i)
                    
                self.figures[title] = fig
                
            # Pass the automatically extracted Rs and Rtot values to the table
            try:
                self._generate_channel_table(ch_num, cell_id, hum_30, hum_50_init, hum_70, hum_90, extracted_rs, extracted_rtot)
            except Exception as e:
                print(f"  -> ERROR Generating Table for CH{ch_num}: {e}")
                traceback.print_exc()
            
        # ADDED: Call the new comprehensive final summary table function before outputting
        self._generate_final_summary_table()
        
        print(f"\n--- Outputting {len(self.figures)} Objects to Browser ---")
        
        # CONFIG FOR MOVABLE TEXT
        base_plotly_config = {
            'toImageButtonOptions': {'format': 'svg', 'scale': 1},
            'editable': True,
            'edits': {
            'annotationPosition': True,
            'annotationTail': False,
            'annotationText': False,
            'axisTitleText': False,
            'titleText': False,
            'legendPosition': True,
            'legendText': False,
            'shapePosition': False 
            }
        }
        
        for title, fig in self.figures.items():
            safe_filename = title.replace(')', '').replace('(', '').replace('°C', 'C').replace('/', '_').replace(' ', '_')
            base_plotly_config['toImageButtonOptions']['filename'] = safe_filename
            fig.show(config=base_plotly_config)

    def _generate_channel_table(self, ch_num, cell_id, df_30, df_50, df_70, df_90, rs_val, rtot_val):
        j_values = {30: "-", 50: "-", 70: "-", 90: "-"}
        for hum, df_hum in zip([30, 50, 70, 90], [df_30, df_50, df_70, df_90]):
            if not df_hum.empty:
                v_arr, j_arr = df_hum.iloc[0]['Voltage'], df_hum.iloc[0]['CurrentDensity']
                if v_arr.size > 0:
                    # Find the current density at the point closest to 1.3V
                    j_at_1_3V = j_arr[np.argmin(np.abs(v_arr - 1.3))]
                    j_values[hum] = f"{j_at_1_3V:.2f}"
                    
        # MODIFIED: Save a clean copy of the j_values for the final summary table before adding HTML tags
        clean_j_values = j_values.copy()
        
        # FIX: Find the highest negative j value (most negative) and prepare it for underlining
        numeric_j_values = {}
        for hum, val_str in j_values.items():
            try:
                val_float = float(val_str)
                # We only care about negative values for underlining
                if val_float < 0:
                    numeric_j_values[hum] = val_float
            except ValueError:
                continue # Skip non-numeric entries like "-"
        # If there are any negative values, find the key (humidity) of the most negative one
        if numeric_j_values:
            max_neg_hum = min(numeric_j_values, key=numeric_j_values.get)
            # Apply HTML underline tag to the string value in the original dictionary
            j_values[max_neg_hum] = f"<u>{j_values[max_neg_hum]}</u>"
            
        # FIX: Converted all headers to have units on the second row.
        headers = [
            ["<b>Cell</b>", ""], 
            ["<b>Testbench</b>", ""], 
            ["<b>Area</b>", "<i>(cm²)</i>"], 
            ["<b>OCV (100% H₂)</b>", "<i>(V)</i>"], 
            ["<b>R<sub>s</sub></b>", "<i>(mΩ·cm²)</i>"], 
            ["<b>R<sub>pol</sub></b>", "<i>(mΩ·cm²)</i>"], 
            ["<b>30% humidity @ 1,3V EC-Mode</b>", "<i>(j / A/cm²)</i>"], 
            ["<b>50% humidity @ 1,3V EC-Mode</b>", "<i>(j / A/cm²)</i>"], 
            ["<b>70% humidity @ 1,3V EC-Mode</b>", "<i>(j / A/cm²)</i>"], 
            ["<b>90% humidity @ 1,3V EC-Mode</b>", "<i>(j / A/cm²)</i>"]
        ]
        
        true_ocv = self.channels[ch_num].get('true_ocv', None)
        ocv_val = f"{true_ocv:.2f}" if true_ocv is not None else "-"
        
        # Calculate resistances and correctly scale them by Area * 1000 for mΩ·cm² 
        try:
            rs_numeric = float(rs_val) * self.channels[ch_num]['area'] * 1000
            formatted_rs = f"{rs_numeric:.2f}"
        except: formatted_rs, rs_numeric = "-", None
            
        try:
            rtot_numeric = float(rtot_val) * self.channels[ch_num]['area'] * 1000
            formatted_rtot = f"{rtot_numeric:.2f}"
        except: formatted_rtot, rtot_numeric = "-", None
            
        if rs_numeric is not None and rtot_numeric is not None:
            formatted_rpol = f"{(rtot_numeric - rs_numeric):.2f}"
        else: formatted_rpol = "-"
        
        # MODIFIED: Save Rs, Rpol, and clean j values to channel dictionary for final summary table
        self.channels[ch_num]['summary_data'] = {
            'rs': formatted_rs,
            'rpol': formatted_rpol,
            'j_vals': clean_j_values
        }
        
        clean_cell_id = cell_id.strip()
        clean_testbench = self.testbench.strip()
        
        row_data = [
            [clean_cell_id], [clean_testbench], [f"{self.channels[ch_num]['area']:.2f}"], [ocv_val], 
            [formatted_rs], [formatted_rpol],
            [j_values[30]], [j_values[50]], [j_values[70]], [j_values[90]]
        ]
        
        fig = go.Figure(data=[go.Table(
            columnwidth=[0.4, 0.8, 0.4, 0.8, 0.5, 0.5, 1.6, 1.6, 1.6, 1.6],
            header=dict(
                values=headers, 
                fill_color='#E1F0FA', 
                font=dict(color='#212F3D', size=15, family="Arial"), 
                align='center', 
                line_color='#D5D8DC', 
                line_width=1,
                height=30 
            ), 
            cells=dict(
                values=row_data, 
                fill_color='white', 
                font=dict(color='#2C3E50', size=15, family="Arial"), 
                align='center', 
                line_color='#D5D8DC', 
                line_width=1,
                height=30
            )
        )])
        
        title = f"Performance Summary Table for {clean_cell_id} (CH{ch_num})"
        
        fig.update_layout(
            title=dict(text=title, font=dict(family="Arial", size=16, color="black")), 
            height=160, 
            width=1600, 
            margin=dict(t=35, b=0, l=0, r=0), 
            paper_bgcolor='rgba(0,0,0,0)' 
        )
        self.figures[title] = fig

    # ADDED: New method to generate the final overarching summary table for all channels
    def _generate_final_summary_table(self):
        headers = [
            ["<b>Cell</b>", ""], 
            ["<b>R<sub>s</sub> @ (100% H₂)</b>", "<i>(mΩ·cm²)</i>"], 
            ["<b>R<sub>pol</sub> @ (100% H₂)</b>", "<i>(mΩ·cm²)</i>"], 
            ["<b>30% humidity @ 1,3V EC-Mode</b>", "<i> j/(A/cm²)</i>"], 
            ["<b>OCV @ 30% humidity</b>", "<i>(V)</i>"],
            ["<b>50% humidity @ 1,3V EC-Mode</b>", "<i> j/(A/cm²)</i>"], 
            ["<b>OCV @ 50% humidity</b>", "<i>(V)</i>"],
            ["<b>70% humidity @ 1,3V EC-Mode</b>", "<i> j/(A/cm²)</i>"], 
            ["<b>OCV @ 70% humidity</b>", "<i>(V)</i>"],
            ["<b>90% humidity @ 1,3V EC-Mode</b>", "<i> j/(A/cm²)</i>"], 
            ["<b>OCV @ 90% humidity</b>", "<i>(V)</i>"]
        ]
        
        row_data = [[] for _ in range(11)]
        
        for ch_num in range(1, 5):
            ch = self.channels[ch_num]
            cell_id = ch['id'].strip()
            
            # Skip channels that have no data
            if cell_id == f'Unknown_CH{ch_num}' and not ch.get('summary_data'):
                continue 
                
            sum_data = ch.get('summary_data', {})
            ocv_data = ch.get('ocv_by_hum', {})
            j_vals = sum_data.get('j_vals', {30: '-', 50: '-', 70: '-', 90: '-'})
            
            row_data[0].append(cell_id)
            row_data[1].append(sum_data.get('rs', '-'))
            row_data[2].append(sum_data.get('rpol', '-'))
            row_data[3].append(j_vals.get(30, '-'))
            row_data[4].append(ocv_data.get(30, '-'))
            row_data[5].append(j_vals.get(50, '-'))
            row_data[6].append(ocv_data.get(50, '-'))
            row_data[7].append(j_vals.get(70, '-'))
            row_data[8].append(ocv_data.get(70, '-'))
            row_data[9].append(j_vals.get(90, '-'))
            row_data[10].append(ocv_data.get(90, '-'))
            
        # If no valid channels were found, abort building table
        if not row_data[0]: return 
        
        fig = go.Figure(data=[go.Table(
            columnwidth=[0.5, 0.8, 0.8, 1.6, 1.4, 1.6, 1.4, 1.6, 1.4, 1.6, 1.4],
            header=dict(
                values=headers, 
                fill_color='#E1F0FA', 
                font=dict(color='#212F3D', size=15, family="Arial"), 
                align='center', 
                line_color='#D5D8DC', 
                line_width=1,
                height=30 
            ), 
            cells=dict(
                values=row_data, 
                fill_color='white', 
                font=dict(color='#2C3E50', size=15, family="Arial"), 
                align='center', 
                line_color='#D5D8DC', 
                line_width=1,
                height=30
            )        
            )])
        
        title = f"Comprehensive Final Summary - All Channels"
        
        fig.update_layout(
            title=dict(text=title, font=dict(family="Arial", size=18, color="black")), 
            height=400, 
            width=1600, 
            margin=dict(t=45, b=10, l=0, r=0), 
            paper_bgcolor='rgba(0,0,0,0)' 
        )
        self.figures[title] = fig

    def _create_base_figure(self, title):
        fig = go.Figure()
        fig.update_layout(
            title=dict(text=title, font=dict(family="Arial", size=22, color="black")), 
            template="plotly_white", 
            height=800, 
            width=1100, 
            hovermode="closest",
            margin=dict(l=80, r=80, t=80, b=80), 
            legend=dict(
                x=0.98, y=0.98, xanchor="right", yanchor="top", 
                bordercolor="#888", borderwidth=1, 
                bgcolor="rgba(255,255,255,0.9)", font=dict(family="Arial", size=14, color="#333"), 
                itemclick="toggle", itemdoubleclick="toggleothers"
            ),
            xaxis=dict(
                title=dict(text="Current Density (j) / A·cm<sup>-2</sup>", font=dict(size=18)), 
                tickfont=dict(size=14), showgrid=True, gridcolor="#E5E5E5", zerolinecolor="#888"
            ),
            yaxis=dict(
                title=dict(text="Voltage (U) / V", font=dict(size=18)), 
                tickfont=dict(size=14), showgrid=True, gridcolor="#E5E5E5", zerolinecolor="#888"
            ),
            # TOGGLE BUTTON: Moved to bottom right outside the plot to prevent overlapping on download
            updatemenus=[
                dict(
                    type="buttons",
                    direction="left",
                    x=1.0, y=-0.15, # Safely below the x-axis 
                    xanchor="right", yanchor="top",
                    showactive=False,
                    buttons=[
                        dict(
                            label="Toggle 1.3V Line",
                            method="relayout",
                            args=[{"shapes[0].visible": False, "annotations[0].visible": False}],
                            args2=[{"shapes[0].visible": True, "annotations[0].visible": True}],
                        )
                    ]
                )
            ]
        )
        
        # NOTE: This MUST be shapes[0]
        fig.add_hline(y=1.3, line_dash="dot", line_color="firebrick", line_width=2, opacity=0.8)
        
        # NOTE: This MUST be annotations[0]
        fig.add_annotation(
            x=0.02, y=1.31, 
            xref="paper", yref="y", 
            text="Thermoneutral Point = 1.3 V", 
            showarrow=False, font=dict(color="firebrick", size=14), 
            xanchor="left", yanchor="bottom",
            bgcolor="rgba(255,255,255,0.95)", 
            bordercolor="firebrick",          
            borderwidth=1,
            borderpad=4
        )
        
        # This is shape[1]
        fig.add_vline(x=0.0, line_width=1, line_color="black")
        return fig

    def _add_trace(self, fig, row, name, color_idx):
        color = self.colors[color_idx % len(self.colors)]
        fig.add_trace(go.Scatter(
            x=row['CurrentDensity'], y=row['Voltage'], name=name, mode='lines', 
            line=dict(width=3, color=color),
            hovertemplate=f"<b>{name}</b><br>j: %{{x:.3f}} A/cm²<br>U: %{{y:.3f}} V<extra></extra>"
        ))


In [20]:

SESSION_FILE = "pol_analyzer_session.json"

def main():
    try:
        from ctypes import windll
        windll.shcore.SetProcessDpiAwareness(1)
    except Exception: 
        pass

    try:
        import customtkinter as ctk
    except ImportError:
        print("\n[!] ERROR: CustomTkinter is not installed.")
        return

    # Palette
    bg_color = ("#FAFAFA", "#09090B")         
    card_color = ("#FFFFFF", "#121214")       
    border_color = ("#E4E4E7", "#27272A")     
    text_main = ("#09090B", "#FAFAFA")        
    text_muted = ("#71717A", "#A1A1AA")       
    baby_blue = ("#38BDF8", "#38BDF8")        
    baby_hover = ("#0EA5E9", "#0EA5E9")       
    btn_neutral = ("#18181B", "#FAFAFA")      
    btn_neutral_hover = ("#27272A", "#E4E4E7")
    btn_neutral_text = ("#FAFAFA", "#09090B") 
    btn_light = ("#F4F4F5", "#27272A")        
    btn_light_hover = ("#E4E4E7", "#3F3F46")
    danger_red = ("#EF4444", "#EF4444")

    ctk.set_appearance_mode("dark") 

    app = ctk.CTk(fg_color=bg_color)
    app.title("KingKev Pol/SOEC Analyzer")
    app.geometry("960x620")
    app.minsize(850, 580)
    
    folders = []
    list_labels = []

    # TOP BAR
    top_bar = ctk.CTkFrame(app, fg_color="transparent")
    top_bar.pack(fill="x", pady=(20, 0))

    def toggle_theme():
        if ctk.get_appearance_mode() == "Dark":
            ctk.set_appearance_mode("Light")
            theme_btn.configure(text="🌙 Dark Mode")
        else:
            ctk.set_appearance_mode("Dark")
            theme_btn.configure(text="☀️ Light Mode")

    theme_btn = ctk.CTkButton(top_bar, text="☀️ Light Mode", command=toggle_theme,
                              fg_color="transparent", hover_color=btn_light_hover, text_color=text_muted,
                              font=("Segoe UI", 12, "bold"), width=40, corner_radius=12)
    theme_btn.pack(anchor="center")

    # HEADER
    header_frame = ctk.CTkFrame(app, fg_color="transparent")
    header_frame.pack(fill="x", pady=(15, 20))
    ctk.CTkLabel(header_frame, text="SOEC Pol Analyzer", font=("Segoe UI", 52, "bold"), text_color=text_main).pack(anchor="center")
    ctk.CTkLabel(header_frame, text="KingKev Pipeline • Ali Obeidat", font=("Segoe UI", 15, "bold"), text_color=baby_blue).pack(anchor="center", pady=(4, 0))

    # TRAY
    card = ctk.CTkFrame(app, fg_color=card_color, corner_radius=16, border_width=1, border_color=border_color)
    card.pack(fill="x", padx=160, pady=(0, 20))

    card_toolbar = ctk.CTkFrame(card, fg_color="transparent")
    card_toolbar.pack(fill="x", padx=25, pady=(15, 5))
    ctk.CTkLabel(card_toolbar, text="Measurement Directories", font=("Segoe UI", 14, "bold"), text_color=text_main).pack(side="left")

    def save_session():
        if not folders: return
        with open(SESSION_FILE, 'w') as f: json.dump(folders, f)
        messagebox.showinfo("Saved", "Session saved!")

    def load_session():
        if not os.path.exists(SESSION_FILE): return
        with open(SESSION_FILE, 'r') as f: saved = json.load(f)
        folders.clear()
        folders.extend([p for p in saved if os.path.exists(p)]) 
        refresh_list()

    def clear_folders():
        folders.clear()
        refresh_list()

    mini_btn_style = {"fg_color": btn_light, "hover_color": btn_light_hover, "font": ("Segoe UI", 11, "bold"), "height": 26, "corner_radius": 8}
    ctk.CTkButton(card_toolbar, text="Clear", command=clear_folders, text_color=danger_red, width=50, **mini_btn_style).pack(side="right", padx=(5, 0))
    ctk.CTkButton(card_toolbar, text="Load", command=load_session, text_color=text_main, width=50, **mini_btn_style).pack(side="right", padx=5)
    ctk.CTkButton(card_toolbar, text="Save", command=save_session, text_color=text_main, width=50, **mini_btn_style).pack(side="right", padx=5)

    scroll_list = ctk.CTkScrollableFrame(card, fg_color="transparent", height=160, scrollbar_button_color=btn_light, scrollbar_button_hover_color=baby_blue)
    scroll_list.pack(fill="x", expand=False, padx=15, pady=(0, 15))

    def remove_individual_folder(folder_path):
        if folder_path in folders:
            folders.remove(folder_path)
            refresh_list()

    def refresh_list():
        for lbl in list_labels: lbl.destroy()
        list_labels.clear()
        if not folders:
            lbl = ctk.CTkLabel(scroll_list, text="Pipeline is empty. Click '+ Add Folders' below.", font=("Segoe UI", 14), text_color=text_muted)
            lbl.pack(pady=45)
            list_labels.append(lbl)
        else:
            for f in folders:
                row = ctk.CTkFrame(scroll_list, fg_color="transparent")
                row.pack(fill="x", pady=2)
                icon = ctk.CTkLabel(row, text="📁", font=("Segoe UI", 16), text_color=baby_blue)
                icon.pack(side="left", padx=(5, 10))
                lbl = ctk.CTkLabel(row, text=os.path.basename(f), font=("Consolas", 13), text_color=text_main)
                lbl.pack(side="left")
                del_btn = ctk.CTkButton(row, text="✕", width=28, height=28, fg_color="transparent", hover_color=btn_light, text_color=text_muted, command=lambda path=f: remove_individual_folder(path))
                del_btn.pack(side="right")
                list_labels.append(row)
    refresh_list() 

    # BOTTOM CONTROLS
    controls_frame = ctk.CTkFrame(app, fg_color="transparent")
    controls_frame.pack(fill="x", pady=(0, 20))

    nyquist_mode_var = ctk.BooleanVar(value=False)
    jc_switch = ctk.CTkSwitch(controls_frame, text="Show Nyquist Plots (Inductive Tails)", variable=nyquist_mode_var,
                              font=("Segoe UI", 13, "bold"), text_color=text_main, progress_color=baby_blue, button_color="#FFFFFF")
    jc_switch.pack(anchor="center", pady=(0, 30))

    action_btn_frame = ctk.CTkFrame(controls_frame, fg_color="transparent")
    action_btn_frame.pack(anchor="center")

    def add_folders():
        added = False
        while True:
            f = filedialog.askdirectory(title="Select Folder (Click Cancel when done)")
            if not f: break
            if f not in folders:
                folders.append(f)
                added = True
        if added: refresh_list()

    def start_analysis():
        if not folders: messagebox.showwarning("Missing Data", "Please add at least one folder.")
        else: app.quit()

    app.protocol("WM_DELETE_WINDOW", lambda: (folders.clear(), app.quit()))

    ctk.CTkButton(action_btn_frame, text="+ Add Folders", command=add_folders, fg_color=btn_neutral, hover_color=btn_neutral_hover, text_color=btn_neutral_text, font=("Segoe UI", 15, "bold"), corner_radius=24, width=150, height=48).pack(side="left", padx=10)
    ctk.CTkButton(action_btn_frame, text="Run Pipeline", command=start_analysis, fg_color=baby_blue, hover_color=baby_hover, text_color="#09090B", font=("Segoe UI", 15, "bold"), corner_radius=24, width=160, height=48).pack(side="left", padx=10)

    app.mainloop()

    nyquist_mode = nyquist_mode_var.get()
    app.destroy()

    if folders:
        print(f"\nProcessing {len(folders)} Folder(s):")
        config = {"custom_colors": px.colors.qualitative.Plotly}
        analyzer = KingKevAnalyzer(folders, config, show_nyquist=nyquist_mode)
        analyzer.parse_master_log()
        analyzer.parse_pol_files()
        analyzer.generate_analyst_plots()
    else: 
        print("Analysis Cancelled. Exiting.")

if __name__ == '__main__':
    main()


Analysis Cancelled. Exiting.
